# Lab 5: Fine-tuning a coding agent with SFT

## Notebook 1: The task, and preparing the data

### The task in one sentence

Given a natural-language programming instruction, generate a correct, self-contained
Python solution.

### Two datasets, two roles

**Training / validation** comes from
[`bigcode/self-oss-instruct-sc2-exec-filter-50k`](https://huggingface.co/datasets/bigcode/self-oss-instruct-sc2-exec-filter-50k)
(ODC-BY). It is a self-generated instruction corpus (built with StarCoder2 via
Self-OSS-Instruct) that was **execution-filtered**: only examples whose response code ran
and passed its own tests were kept. We take a shuffled **10,000-example subset** so the
single-epoch job fits a short workshop window, and format each row as a `prompt`/`completion`
pair.

**Testing** uses [HumanEval](https://huggingface.co/datasets/openai_humaneval) (MIT), 164
hand-written problems, each shipping a canonical unit-test suite. Using HumanEval as the
held-out set makes `pass@1` a real execution measurement (notebook 3 runs the model's code
against the tests) and guarantees no overlap with the training data.

### Why fine-tune when the base model already writes Python?

A 4B base model can write Python; the question is whether it does so **reliably and in a
runnable shape** rather than as chatty prose around a snippet. SFT on execution-verified
examples pushes the model toward emitting a solution directly, which is exactly what
`pass@1` rewards. The improvement to look for in notebook 3 is a higher fraction of
generated solutions that execute correctly, measured identically on the base and
fine-tuned models.

> **Single dataset, on purpose.** With ~10K examples and one epoch, one focused task
> maximises the visible base-vs-fine-tuned delta. Mixing in agent-trajectory or reasoning
> data would dilute the signal. Agentic behaviour (tool use, multi-step planning) comes at
> inference time from the coding-agent framework that orchestrates the model, not from this
> SFT step.

### Install requirements

In [ ]:
%pip install -r requirements.txt

#### Setup and dependencies

Standard SageMaker boilerplate, no task-specific logic: it resolves the execution role,
the default bucket and the bucket prefix, and opens the boto3 clients. This notebook reuses
`s3_client`, `bucket_name` and `default_prefix` in the upload cell. Notebooks 2 and 4 repeat
this cell.

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")

### First: what is `codeagent.py`?

Every notebook in this lab starts with `import codeagent as C`. Like `contractnli.py` in
Lab 1, it holds only the mechanical parts, loading the datasets and rendering the prompt,
while calling a model, executing its code and scoring it stay in the notebooks.

| Call | Returns | Used for |
|---|---|---|
| `C.load_subset(n, seed)` | a `datasets.Dataset` of `n` rows | the shuffled BigCode training subset |
| `C.split(ds, val)` | `(train, val)` datasets | a disjoint validation slice |
| `C.build_prompt(task)` | one string | the whole request the model reads |
| `C.completion_for(row)` | one string | the target solution (BigCode `response`) |
| `C.load_humaneval()` | a `datasets.Dataset` | the 164-problem HumanEval test set |
| `C.humaneval_task(row)` | one string | the prompt shown to the model |

One constant matters too: `C.INSTRUCTION`, the template `build_prompt` fills in.

### Load the training subset

`load_subset` shuffles with a fixed seed and takes the first `SUBSET_SIZE` rows, so the
run is repeatable. `split` then carves a small validation slice off it. HumanEval is the
test set, so nothing from training is ever scored.

In [ ]:
import codeagent as C
from config import BASE_MODEL_ID, DATASET_PREFIX, SUBSET_SIZE, VAL_SIZE

base_model_id = BASE_MODEL_ID   # "huggingface-reasoning-qwen3-4b"

subset = C.load_subset(SUBSET_SIZE, seed=42)
train_ds, val_ds = C.split(subset, val=VAL_SIZE)
test_ds = C.load_humaneval()

print(f"train {len(train_ds)} | val {len(val_ds)} | test (HumanEval) {len(test_ds)}")

#### What one training row looks like

The BigCode row carries an `instruction` (the task) and a `response` (execution-verified
working code, usually with a short explanation and tests). We train on those two fields.

In [ ]:
row = train_ds[0]
print("fields:", list(row.keys()), "\n")
print("INSTRUCTION:\n", row["instruction"][:400], "\n")
print("RESPONSE (target completion):\n", row["response"][:400])

### The prompt

Everything the model reads is one string: the template below with the task filled in.
Keeping it to "code only, inside a single ```python block" keeps the output short and
runnable, which is what `pass@1` executes.

It is defined **once**, in `codeagent.py`, and used by every notebook, data prep here, the
evaluation in notebook 3, and the serving check in notebook 4. That is the discipline that
keeps the trained prompt byte-identical to the inference prompt.

In [ ]:
print("=" * 70, "\nINSTRUCTION TEMPLATE\n", "=" * 70, sep="")
print(C.INSTRUCTION)
print("\n", "=" * 70, "\nRENDERED FOR ONE TASK\n", "=" * 70, sep="")
print(C.build_prompt(train_ds[0]["instruction"]))

To experiment with the wording, set `C.INSTRUCTION` here and re-run the record build
below; every notebook then picks up your version:

```python
C.INSTRUCTION = """...your wording, keeping {task}..."""
```

### Build the records

**Train / val** use the `prompt`/`completion` shape serverless SFT accepts: the instruction
(wrapped by the template) in `prompt`, the target solution in `completion`.

**Test** uses `query`/`response` (the `gen_qa` format the managed scorer in notebook 3
requires), and carries two extra fields the `pass@1` scorer needs to execute the model's
code: the problem's `test` suite and its `entry_point`.

> **Why the whole instruction lives in `query`.** The evaluation container does not reliably
> pass a separate `system` field through to the model (Lab 1 notebook 3 documents this). So
> the entire prompt is folded into `query`, and `system` is unused. The `test` and
> `entry_point` fields ride alongside for the scorer, not the model.

In [ ]:
def make_records(ds):
    """Training records: one prompt/completion pair per example."""
    return [{"prompt": C.build_prompt(r["instruction"]),
             "completion": C.completion_for(r)}
            for r in ds]


def make_test_records(ds):
    """HumanEval evaluation records. `query` is the prompt string; `test` and
    `entry_point` let the pass@1 scorer execute the model's code."""
    return [{"query": C.build_prompt(C.humaneval_task(r)),
             "response": C.humaneval_reference(r),
             "test": r["test"],
             "entry_point": r["entry_point"],
             "task_id": r["task_id"]}
            for r in ds]


records = {"train": make_records(train_ds),
           "val": make_records(val_ds),
           "test": make_test_records(test_ds)}

for name, rows in records.items():
    field = "query" if name == "test" else "prompt"
    avg = sum(len(r[field]) for r in rows) // len(rows)
    print(f"{name:5s}: {len(rows):5d} records, avg prompt {avg:5d} chars (~{avg // 4} tokens)")

One built training record, and one test record:

In [ ]:
tr = records["train"][0]
print("train fields:", list(tr))
print(f"  prompt:     {len(tr['prompt']):,} chars")
print(f"  completion: {len(tr['completion']):,} chars")

te = records["test"][0]
print("\ntest fields:", list(te))
print(f"  entry_point: {te['entry_point']}  |  task_id: {te['task_id']}")
print(f"  test suite:  {len(te['test'])} chars")

#### Write to disk and upload to Amazon S3

Each split is written as `./sft_data/<split>/dataset.jsonl`, one JSON object per line, the
shape `DataSet.create` validates before registering it. Each file lands at
`s3://<bucket>/[<prefix>/]datasets/self-oss-code-sft/<split>/dataset.jsonl`.

In [ ]:
import json
import pathlib
import shutil

local = pathlib.Path("./sft_data")
if local.exists():
    shutil.rmtree(local)

for name, rows in records.items():
    d = local / name
    d.mkdir(parents=True, exist_ok=True)
    with open(d / "dataset.jsonl", "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

input_path = (f"{default_prefix}/datasets/{DATASET_PREFIX}" if default_prefix
              else f"datasets/{DATASET_PREFIX}")

s3_paths = {}
for name in records:
    key = f"{input_path}/{name}/dataset.jsonl"
    s3_client.upload_file(str(local / name / "dataset.jsonl"), bucket_name, key)
    s3_paths[name] = f"s3://{bucket_name}/{key}"
    print(s3_paths[name])

#### Register the datasets

`DataSet.create` writes a registry entry pointing at the S3 object. Notebook 2 calls
`DataSet.get(name=...)` and hands the result to `SFTTrainer`, which records the entry's name
and version rather than an S3 URI. `wait=True` blocks until each import reaches `Available`,
and raises on `ImportFailed`.

| dataset | technique | consumed by |
|---|---|---|
| `self-oss-code-sft-train` | `SFT` | notebook 2, `training_dataset=` |
| `self-oss-code-sft-val` | `SFT` | notebook 2, `validation_dataset=` |
| `self-oss-code-sft-test` | none | notebook 3, `dataset=` |

> **Note:** `create` imports the next major version and always resolves a bare name to the
> latest, but the entry stores only a bucket and key with no checksum, and the upload cell
> always writes the same key. Re-upload before you re-register.

In [ ]:
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.dataset_utils import CustomizationTechnique


def register(name, source, technique=None):
    kwargs = dict(name=name, source=source, wait=True)
    if technique is not None:
        kwargs["customization_technique"] = technique
    ds = DataSet.create(**kwargs)
    print(f"created dataset: {name}")
    return ds


training_dataset = register(f"{DATASET_PREFIX}-train", s3_paths["train"], CustomizationTechnique.SFT)
val_dataset = register(f"{DATASET_PREFIX}-val", s3_paths["val"], CustomizationTechnique.SFT)
test_dataset = register(f"{DATASET_PREFIX}-test", s3_paths["test"])

### What you built

`self-oss-code-sft-train`, `-val` and `-test` (HumanEval), registered and ready. Notebooks 2
and 3 look them up by name.

Continue to **notebook 2** to run the serverless LoRA fine-tuning job.